In [20]:
from pathlib import Path
import json
import pandas as pd

DATA_PATH = Path("all-sources.json")
with DATA_PATH.open(encoding="utf-8") as f:
    payload = json.load(f)

boats = payload["data"]
metadata = payload.get("metadata", {})
df = pd.DataFrame(boats)
df.head()


,Source,DocumentID,ListingTitle,BeamMeasure,ItemReceivedDate,TotalEnginePowerQuantity,Price,BoatLocation,Model,FuelTankCapacityMeasure,...,Engines,ModelYear,OriginalPrice,MakeString,LengthOverall,NominalLength,GeneralBoatDescription,LastModificationDate,AdditionalDetailDescription,Images
0,service,9785609,New arrival,10.83 ft,2025-05-07,600 hp,114900.00 USD,"{'BoatCityName': 'Bradenton', 'BoatCountryID':...",320 Sport Yacht,180|gallon,...,"[{'Make': 'MerCruiser', 'Model': '350 Mag MPI ...",2013,114900 USD,Monterey,33.83 ft,32 ft,"[<p>NEW Listing- This is a low hour, fully loa...",2025-11-29,[<strong>Standards</strong><br><p>Hull & Deck<...,"{'Priority': '0', 'Caption': '2013 Monterey 32..."
1,service,9758424,New arrival,12.75 ft,2025-04-14,700 hp,229000.00 USD,"{'BoatCityName': 'Marco Island', 'BoatCountryI...",320 Vantage,285|gallon,...,"[{'Make': 'Mercury', 'Model': 'Verado', 'Fuel'...",2018,229000 USD,Boston Whaler,32.67 ft,32 ft,[<p><br /></p><p>Well-Maintained Boston Whaler...,2025-11-29,[<strong>Disclaimer</strong><br><p>The Company...,"{'Priority': '0', 'Caption': '2018 Boston Whal..."
2,service,9973157,2021 Boston Whaler 350 REALM,None,2025-10-08,900 hp,479000.00 USD,"{'BoatCityName': 'Clearwater', 'BoatCountryID'...",350 Realm,|,...,"[{'Make': 'MERCURY', 'Model': '300 V8', 'Drive...",2021,479000 USD,Boston Whaler,35 ft,35 ft,[<p><strong><em>2021 Boston Whaler 350 Realm</...,2025-11-29,NaN,"{'Priority': '0', 'Caption': '', 'Uri': 'https..."
3,service,9925765,NaN,9.25 ft,2025-08-27,400 hp,62960.00 USD,"{'BoatCityName': 'Miami', 'BoatCountryID': 'US...",2670 Denali LS,188|gallon,...,"[{'Make': 'Yamaha', 'Model': 'F200TXRD', 'Fuel...",2005,62960 USD,Pursuit,28.42 ft,26 ft,"[<p id=""isPasted""><strong>Financing available....",2025-11-29,[<strong>Disclaimer</strong><br><p>The Company...,"{'Priority': '0', 'Caption': '2005 Pursuit 267..."
4,service,10020760,New arrival,9.5 ft,2025-11-21,450 hp,35000.00 USD,"{'BoatCityName': 'Islamorada', 'BoatCountryID'...",270,175|gallon,...,"[{'Make': 'Yamaha', 'Model': 'F225', 'Fuel': '...",2007,35000 USD,Cobia,27 ft,27 ft,[<p>The **Cobia 27 Walkaround 2007** is a vers...,2025-11-29,[<strong>Disclaimer</strong><br><p>The Company...,"{'Priority': '0', 'Caption': '2007 Cobia 270 b..."


In [21]:
df.to_csv("all-sources.csv", index=False)
df.columns

Index(['Source', 'DocumentID', 'ListingTitle', 'BeamMeasure',
       'ItemReceivedDate', 'TotalEnginePowerQuantity', 'Price', 'BoatLocation',
       'Model', 'FuelTankCapacityMeasure', 'FuelTankCountNumeric', 'Engines',
       'ModelYear', 'OriginalPrice', 'MakeString', 'LengthOverall',
       'NominalLength', 'GeneralBoatDescription', 'LastModificationDate',
       'AdditionalDetailDescription', 'Images'],
      dtype='object')

In [8]:
# Examine sample text fields to identify cleaning needs
sample_idx = 0
sample_boat = boats[sample_idx]

print("Sample boat record (selected fields):")
text_fields = ['Description', 'MakeString', 'Model', 'Price', 'BoatName']
for field in text_fields:
    if field in sample_boat:
        value = sample_boat[field]
        print(f"\n{field}:")
        print(f"  Type: {type(value)}")
        print(f"  Value: {str(value)[:200]}...")  # First 200 chars

Sample boat record (selected fields):

MakeString:
  Type: <class 'str'>
  Value: Monterey...

Model:
  Type: <class 'str'>
  Value: 320 Sport Yacht...

Price:
  Type: <class 'str'>
  Value: 114900.00 USD...


In [9]:
# Check for HTML tags and special characters in text fields
import re
from html import unescape

def check_for_html_tags(text):
    """Detect if text contains HTML tags"""
    if not isinstance(text, str):
        return False
    return bool(re.search(r'<[^>]+>', text))

def check_for_entities(text):
    """Detect HTML entities"""
    if not isinstance(text, str):
        return False
    return bool(re.search(r'&[a-z]+;|&#\d+;', text, re.IGNORECASE))

# Check multiple records
fields_to_check = ['Description', 'MakeString', 'Model', 'BoatName', 'AdditionalDetailDescription']
results = {field: {'has_html': 0, 'has_entities': 0, 'total': 0} for field in fields_to_check}

for boat in boats[:100]:  # Check first 100
    for field in fields_to_check:
        if field in boat and boat[field]:
            value = str(boat[field])
            results[field]['total'] += 1
            if check_for_html_tags(value):
                results[field]['has_html'] += 1
            if check_for_entities(value):
                results[field]['has_entities'] += 1

print("HTML/Entity Detection Results (first 100 records):")
for field, counts in results.items():
    if counts['total'] > 0:
        print(f"\n{field}:")
        print(f"  Records with HTML tags: {counts['has_html']} / {counts['total']}")
        print(f"  Records with entities: {counts['has_entities']} / {counts['total']}")

HTML/Entity Detection Results (first 100 records):

MakeString:
  Records with HTML tags: 0 / 100
  Records with entities: 0 / 100

Model:
  Records with HTML tags: 0 / 100
  Records with entities: 0 / 100

AdditionalDetailDescription:
  Records with HTML tags: 78 / 78
  Records with entities: 0 / 78


In [13]:
# Check all fields in dataset for HTML content
all_fields = set()
for boat in boats[:5]:
    all_fields.update(boat.keys())

print(f"Total unique fields: {len(all_fields)}")
print(f"Fields: {sorted(all_fields)[:20]}")  # Show first 20

# Check for HTML in any string field
html_count_by_field = {}
for boat in boats[:100]:
    for field, value in boat.items():
        if isinstance(value, str) and ('<' in value or '&' in value):
            if field not in html_count_by_field:
                html_count_by_field[field] = 0
            html_count_by_field[field] += 1

print("\n\nFields with potential HTML/entities (first 100 records):")
for field, count in sorted(html_count_by_field.items(), key=lambda x: x[1], reverse=True):
    print(f"  {field}: {count} records")

Total unique fields: 21
Fields: ['AdditionalDetailDescription', 'BeamMeasure', 'BoatLocation', 'DocumentID', 'Engines', 'FuelTankCapacityMeasure', 'FuelTankCountNumeric', 'GeneralBoatDescription', 'Images', 'ItemReceivedDate', 'LastModificationDate', 'LengthOverall', 'ListingTitle', 'MakeString', 'Model', 'ModelYear', 'NominalLength', 'OriginalPrice', 'Price', 'Source']


Fields with potential HTML/entities (first 100 records):


In [14]:
# Show actual content samples from key fields
print("Sample data from key text fields:\n")
sample = boats[0]

for field in ['GeneralBoatDescription', 'AdditionalDetailDescription', 'ListingTitle', 'MakeString', 'Model']:
    if field in sample and sample[field]:
        print(f"{field}:")
        content = str(sample[field])
        print(f"  {content[:200]}...")
        print()

Sample data from key text fields:

GeneralBoatDescription:
  ['<p>NEW Listing- This is a low hour, fully loaded Monterey 320 Sport Yacht. \xa0The boat is powered by twin Mercruiser 350 Mag engines (Which is an upgrade from the standard 5.0L) with Mercruiser Bra...

AdditionalDetailDescription:
  ['<strong>Standards</strong><br><p>Hull & Deck</p><ul> <li>AME Vinylester Resin</li> <li>Anchor Chafe Plate, Stainless Steel</li> <li>Anchor Roller, Stainless Steel</li> <li>Bow & Stern Eyes, Stainles...

ListingTitle:
  New arrival...

MakeString:
  Monterey...

Model:
  320 Sport Yacht...



## Data Cleaning Strategy

Based on the analysis, here's what needs cleaning:

1. **HTML Tags**: Fields like `GeneralBoatDescription` and `AdditionalDetailDescription` contain HTML markup (`<p>`, `<strong>`, `<ul>`, `<li>`, `<br>`, etc.)
2. **Special Characters**: Non-breaking spaces (`\xa0`) and other Unicode characters
3. **Extra Whitespace**: Multiple spaces, newlines after tag removal
4. **List Formatting**: Some fields contain Python list representations

**Recommended Cleaning Approach:**

In [15]:
from bs4 import BeautifulSoup
import re

def clean_text_field(value):
    """
    Clean text by removing HTML tags, special characters, and normalizing whitespace.
    """
    if not value:
        return value
    
    # Convert to string if it's a list (some fields store as single-item lists)
    if isinstance(value, list):
        if len(value) == 0:
            return None
        value = value[0] if len(value) == 1 else ' '.join(str(v) for v in value)
    
    if not isinstance(value, str):
        return value
    
    # Remove HTML tags using BeautifulSoup
    soup = BeautifulSoup(value, 'html.parser')
    text = soup.get_text(separator=' ')
    
    # Replace non-breaking spaces and other unicode whitespace
    text = text.replace('\xa0', ' ')
    text = text.replace('\u200b', '')  # Zero-width space
    text = text.replace('\r', ' ')
    
    # Normalize whitespace (collapse multiple spaces into one)
    text = re.sub(r'\s+', ' ', text)
    
    # Trim leading/trailing whitespace
    text = text.strip()
    
    return text if text else None


def clean_boat_record(boat):
    """
    Clean all text fields in a boat record.
    """
    cleaned = boat.copy()
    
    # Fields that typically contain HTML or need cleaning
    text_fields = [
        'GeneralBoatDescription',
        'AdditionalDetailDescription', 
        'ListingTitle',
        'MakeString',
        'Model',
        'BoatName'
    ]
    
    for field in text_fields:
        if field in cleaned and cleaned[field]:
            cleaned[field] = clean_text_field(cleaned[field])
    
    return cleaned

# Test on a sample
print("BEFORE CLEANING:")
print(f"GeneralBoatDescription: {boats[0].get('GeneralBoatDescription', 'N/A')[:200]}...")
print()

cleaned_sample = clean_boat_record(boats[0])
print("AFTER CLEANING:")
print(f"GeneralBoatDescription: {cleaned_sample.get('GeneralBoatDescription', 'N/A')[:200]}...")
print()
print(f"AdditionalDetailDescription: {cleaned_sample.get('AdditionalDetailDescription', 'N/A')[:200]}...")

BEFORE CLEANING:
GeneralBoatDescription: ['<p>NEW Listing- This is a low hour, fully loaded Monterey 320 Sport Yacht. \xa0The boat is powered by twin Mercruiser 350 Mag engines (Which is an upgrade from the standard 5.0L) with Mercruiser Bravo III drives. \xa0Other options include: Hard-top, Joystick docking control, Radar, GPS/Depth, Digital gauges, Trim tabs, Kohler Generator, AC, Marine Head, Stove top, Microwave, Two Fridges, Convertible forward birth, Rear sunpad, Windlass, Cockpit cover and underwater lights. Recent maintenance on the outdrives.</p><p>Leave the turmoil of civilization behind aboard the 320 Sport Yacht. A perfect weekend getaway, the 320SY is equipped with many of the features of its larger sisters. The helm on the 320SY is unique featuring twin forward facing helm seats and chart storage on the power side. Built in steps in the cabin door allow safe access to the bow where a large sun pad awaits.</p>']...

AFTER CLEANING:
GeneralBoatDescription: NEW Listing- This

In [16]:
# Clean all boat records
print("Cleaning all boat records...")
cleaned_boats = [clean_boat_record(boat) for boat in boats]

# Update the payload
cleaned_payload = payload.copy()
cleaned_payload['data'] = cleaned_boats

# Save to a new file
output_path = Path("all-sources-cleaned.json")
with output_path.open('w', encoding='utf-8') as f:
    json.dump(cleaned_payload, f, indent=2, ensure_ascii=False)

print(f"✓ Cleaned {len(cleaned_boats)} boat records")
print(f"✓ Saved to: {output_path}")

Cleaning all boat records...
✓ Cleaned 2000 boat records
✓ Saved to: all-sources-cleaned.json


In [17]:
# Compare before/after statistics
import random

print("=== BEFORE vs AFTER COMPARISON ===\n")

# Show 3 random examples
samples = random.sample(range(len(boats)), 3)

for idx in samples:
    original = boats[idx]
    cleaned = cleaned_boats[idx]
    
    print(f"Example {idx + 1}:")
    print(f"Make/Model: {cleaned.get('MakeString')} {cleaned.get('Model')}")
    
    # Compare description field
    if 'GeneralBoatDescription' in original and original['GeneralBoatDescription']:
        orig_desc = str(original['GeneralBoatDescription'])
        clean_desc = cleaned.get('GeneralBoatDescription', '')
        
        print(f"\nOriginal length: {len(orig_desc)} chars")
        print(f"Cleaned length: {len(clean_desc)} chars")
        print(f"Cleaned preview: {clean_desc[:150]}...")
    
    print("\n" + "="*80 + "\n")

=== BEFORE vs AFTER COMPARISON ===

Example 1496:
Make/Model: Formula 45 Yacht

Original length: 799 chars
Cleaned length: 413 chars
Cleaned preview: Twin VOLVO IPS 600's both drives updated and over $40,000 invested. Just over 1200 hours with extensive service history - Fresh oil changes with sampl...


Example 713:
Make/Model: Crownline Eclipse E255 SURF

Original length: 1841 chars
Cleaned length: 564 chars
Cleaned preview: ALL TRADES WELCOME FINANCING & SHIPPING AVAILABLE WE WANT YOUR TRADES ALWAYS FRESHWATER USED This wonderful 2019 Crownline E255 Surf is perfect for fa...


Example 885:
Make/Model: Sabre 58 Salon Express

Original length: 12 chars
Cleaned length: 1 chars
Cleaned preview: ....




## Summary

✅ **Cleaning Complete!**

The cleaned dataset has been saved to `all-sources-cleaned.json` with the following improvements:

1. **Removed HTML tags** (`<p>`, `<strong>`, `<ul>`, `<li>`, `<br>`, etc.)
2. **Normalized whitespace** (removed `\xa0`, collapsed multiple spaces)
3. **Converted list fields** to proper strings
4. **Trimmed excess whitespace**

**Key fields cleaned:**
- `GeneralBoatDescription`
- `AdditionalDetailDescription`
- `ListingTitle`
- `MakeString`
- `Model`
- `BoatName`

**Next Steps:**
- Use `all-sources-cleaned.json` for your search chatbot
- Text is now ready for vectorization/embedding
- No HTML interference in search results

In [23]:
summary = {
    "records_in_file": len(boats),
    "metadata": {
        "total_records": metadata.get("total"),
        "page": metadata.get("page"),
        "total_pages": metadata.get("totalPage"),
        "page_limit": metadata.get("limit"),
    },
    "sample_columns": df.columns.tolist(),
}
summary

{'records_in_file': 2000,
 'metadata': {'total_records': 3620,
  'page': 1,
  'total_pages': 2,
  'page_limit': 2000},
 'sample_columns': ['Source',
  'DocumentID',
  'ListingTitle',
  'BeamMeasure',
  'ItemReceivedDate',
  'TotalEnginePowerQuantity',
  'Price',
  'BoatLocation',
  'Model',
  'FuelTankCapacityMeasure',
  'FuelTankCountNumeric',
  'Engines',
  'ModelYear',
  'OriginalPrice',
  'MakeString',
  'LengthOverall',
  'NominalLength',
  'GeneralBoatDescription',
  'LastModificationDate',
  'AdditionalDetailDescription',
  'Images']}

In [27]:
df.columns

Index(['Source', 'DocumentID', 'ListingTitle', 'BeamMeasure',
       'ItemReceivedDate', 'TotalEnginePowerQuantity', 'Price', 'BoatLocation',
       'Model', 'FuelTankCapacityMeasure', 'FuelTankCountNumeric', 'Engines',
       'ModelYear', 'OriginalPrice', 'MakeString', 'LengthOverall',
       'NominalLength', 'GeneralBoatDescription', 'LastModificationDate',
       'AdditionalDetailDescription', 'Images'],
      dtype='object')

In [29]:
DATA_PATH = Path("all-sources-cleaned.json")
with DATA_PATH.open(encoding="utf-8") as f:
    payload = json.load(f)

boats = payload["data"]
metadata = payload.get("metadata", {})
df = pd.DataFrame(boats)
df.to_csv("all-sources-cleaned.csv", index=False)
df.head()

,Source,DocumentID,ListingTitle,BeamMeasure,ItemReceivedDate,TotalEnginePowerQuantity,Price,BoatLocation,Model,FuelTankCapacityMeasure,...,Engines,ModelYear,OriginalPrice,MakeString,LengthOverall,NominalLength,GeneralBoatDescription,LastModificationDate,AdditionalDetailDescription,Images
0,service,9785609,New arrival,10.83 ft,2025-05-07,600 hp,114900.00 USD,"{'BoatCityName': 'Bradenton', 'BoatCountryID':...",320 Sport Yacht,180|gallon,...,"[{'Make': 'MerCruiser', 'Model': '350 Mag MPI ...",2013,114900 USD,Monterey,33.83 ft,32 ft,"NEW Listing- This is a low hour, fully loaded ...",2025-11-29,Standards Hull & Deck AME Vinylester Resin Anc...,"{'Priority': '0', 'Caption': '2013 Monterey 32..."
1,service,9758424,New arrival,12.75 ft,2025-04-14,700 hp,229000.00 USD,"{'BoatCityName': 'Marco Island', 'BoatCountryI...",320 Vantage,285|gallon,...,"[{'Make': 'Mercury', 'Model': 'Verado', 'Fuel'...",2018,229000 USD,Boston Whaler,32.67 ft,32 ft,Well-Maintained Boston Whaler 320 Vantage – Th...,2025-11-29,Disclaimer The Company offers the details of t...,"{'Priority': '0', 'Caption': '2018 Boston Whal..."
2,service,9973157,2021 Boston Whaler 350 REALM,None,2025-10-08,900 hp,479000.00 USD,"{'BoatCityName': 'Clearwater', 'BoatCountryID'...",350 Realm,|,...,"[{'Make': 'MERCURY', 'Model': '300 V8', 'Drive...",2021,479000 USD,Boston Whaler,35 ft,35 ft,"2021 Boston Whaler 350 Realm Extremely clean, ...",2025-11-29,NaN,"{'Priority': '0', 'Caption': '', 'Uri': 'https..."
3,service,9925765,NaN,9.25 ft,2025-08-27,400 hp,62960.00 USD,"{'BoatCityName': 'Miami', 'BoatCountryID': 'US...",2670 Denali LS,188|gallon,...,"[{'Make': 'Yamaha', 'Model': 'F200TXRD', 'Fuel...",2005,62960 USD,Pursuit,28.42 ft,26 ft,Financing available. All service records in ha...,2025-11-29,Disclaimer The Company offers the details of t...,"{'Priority': '0', 'Caption': '2005 Pursuit 267..."
4,service,10020760,New arrival,9.5 ft,2025-11-21,450 hp,35000.00 USD,"{'BoatCityName': 'Islamorada', 'BoatCountryID'...",270,175|gallon,...,"[{'Make': 'Yamaha', 'Model': 'F225', 'Fuel': '...",2007,35000 USD,Cobia,27 ft,27 ft,The **Cobia 27 Walkaround 2007** is a versatil...,2025-11-29,Disclaimer The Company offers the details of t...,"{'Priority': '0', 'Caption': '2007 Cobia 270 b..."


In [30]:
# Load cleaned data and create DataFrame with Make column
cleaned_df = pd.DataFrame(cleaned_boats)

# Extract Make as a separate column (MakeString is the manufacturer)
cleaned_df['Make'] = cleaned_df['MakeString']

# Show the new column
print(f"Total records: {len(cleaned_df)}")
print(f"\nColumns: {cleaned_df.columns.tolist()}")
print(f"\nSample of Make column:")
cleaned_df[['Make', 'Model', 'ModelYear', 'Price']].head(10)

Total records: 2000

Columns: ['Source', 'DocumentID', 'ListingTitle', 'BeamMeasure', 'ItemReceivedDate', 'TotalEnginePowerQuantity', 'Price', 'BoatLocation', 'Model', 'FuelTankCapacityMeasure', 'FuelTankCountNumeric', 'Engines', 'ModelYear', 'OriginalPrice', 'MakeString', 'LengthOverall', 'NominalLength', 'GeneralBoatDescription', 'LastModificationDate', 'AdditionalDetailDescription', 'Images', 'Make']

Sample of Make column:


,Make,Model,ModelYear,Price
0,Monterey,320 Sport Yacht,2013,114900.00 USD
1,Boston Whaler,320 Vantage,2018,229000.00 USD
2,Boston Whaler,350 Realm,2021,479000.00 USD
3,Pursuit,2670 Denali LS,2005,62960.00 USD
4,Cobia,270,2007,35000.00 USD
5,Cobalt,CS23,2019,55000.00 USD
6,Galeon,375 GTO,2024,900000.00 USD
7,Four Winns,318 Vista,2007,69995.00 USD
8,Cobalt,R30,2021,219900.00 USD
9,Cobalt,A36,2018,239000.00 USD


In [31]:
# Export to CSV with Make as an independent column
csv_output_path = Path("all-sources-cleaned.csv")
cleaned_df.to_csv(csv_output_path, index=False, encoding='utf-8')

print(f"✓ Exported {len(cleaned_df)} records to CSV")
print(f"✓ File: {csv_output_path}")
print(f"✓ File size: {csv_output_path.stat().st_size / 1024 / 1024:.2f} MB")
print(f"\nColumns in CSV ({len(cleaned_df.columns)}):")
for i, col in enumerate(cleaned_df.columns, 1):
    print(f"  {i}. {col}")

✓ Exported 2000 records to CSV
✓ File: all-sources-cleaned.csv
✓ File size: 10.17 MB

Columns in CSV (22):
  1. Source
  2. DocumentID
  3. ListingTitle
  4. BeamMeasure
  5. ItemReceivedDate
  6. TotalEnginePowerQuantity
  7. Price
  8. BoatLocation
  9. Model
  10. FuelTankCapacityMeasure
  11. FuelTankCountNumeric
  12. Engines
  13. ModelYear
  14. OriginalPrice
  15. MakeString
  16. LengthOverall
  17. NominalLength
  18. GeneralBoatDescription
  19. LastModificationDate
  20. AdditionalDetailDescription
  21. Images
  22. Make


In [32]:
# Verify the CSV was created correctly
verification_df = pd.read_csv(csv_output_path, nrows=5)
print("CSV Verification - First 5 rows:")
verification_df[['Make', 'Model', 'ModelYear', 'Price']].head()

CSV Verification - First 5 rows:


,Make,Model,ModelYear,Price
0,Monterey,320 Sport Yacht,2013,114900.00 USD
1,Boston Whaler,320 Vantage,2018,229000.00 USD
2,Boston Whaler,350 Realm,2021,479000.00 USD
3,Pursuit,2670 Denali LS,2005,62960.00 USD
4,Cobia,270,2007,35000.00 USD
